<a href="https://colab.research.google.com/github/lilhawkeye2002-ux/notebooks/blob/main/Whisper_Bulk_Transcriber.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Whisper Bulk Transcriber — User Guide

Automatically transcribe **any number of audio or video files** into readable text using OpenAI's Whisper `large-v2` model. No configuration needed — just drop your files in a folder and run.

---

## 🚀 How To Use (3 Steps)

### Step 1 — Run the cell for the first time

Click the **▶ Run** button on the code cell below (or press `Shift + Enter` with the cell selected).

On first run it will:
- Install all required libraries automatically
- Create the upload folder: `/content/bulk_process_audios_here`
- Stop and display this message:
  > *"Directory created. Upload your audio/video files there, then run this cell again."*

---

### Step 2 — Upload your files

Open the **Files panel** on the left sidebar (the folder icon 📁), then navigate to `/content/bulk_process_audios_here`.

**Drag and drop** your audio or video files directly into that folder.

> 💡 **Tip:** You can drop as many files as you like at once — the notebook will process them all in one go.

**Supported formats:**
| Audio | Video |
|-------|-------|
| `.mp3` `.wav` `.m4a` `.flac` | `.mp4` `.mov` `.avi` `.mkv` |
| `.aac` `.ogg` `.wma` `.opus` | `.webm` |
| `.aiff` `.amr` `.au` | |

> 🎬 **Video files are handled automatically** — the audio track is extracted for you before transcription begins.

---

### Step 3 — Run the cell again

Click **▶ Run** once more. The notebook will detect your files and begin transcribing immediately.

---

## 📺 What To Expect

Once files are detected, here is exactly what happens in order:

**① Library check**
> A quick check confirms all required packages are already installed. No waiting — this is instant on re-runs.

**② GPU check**
> The notebook detects whether a GPU is available.
> - ✅ **GPU found** — transcription runs at full speed.
> - ⚠️ **No GPU found** — you'll see a warning explaining that transcription will be *very* slow (roughly 1 line per minute). You'll be asked to confirm before it continues. To enable a free GPU: **Runtime → Change runtime type → Hardware accelerator → T4 GPU**, then re-run.

**③ Model loading**
> The `large-v2` Whisper model loads onto the GPU (or CPU). This takes 20–60 seconds the first time. You'll see:
> ```
> Loading 'large-v2' model on cuda...
> Model 'large-v2' loaded on cuda.
> ```

**④ Per-file transcription**
> Each file is processed one at a time. You'll see a live feed of the transcript as it's being generated, line by line:
> ```
> [1/3] Processing: Interview.mp3
> [00:00.000 --> 00:04.820]  Welcome everyone, today we're going to talk about...
> [00:04.820 --> 00:09.110]  the fundamentals of machine learning and why it matters...
> ```
> Video files will show an extra step:
> ```
> [2/3] Processing: Lecture.mp4
>   Video file detected — extracting audio track via ffmpeg...
>   Audio extracted successfully.
> ```

**⑤ Output files saved**
> After each file finishes, three transcript formats are saved to `/content/audio_transcription/`:
> - **`.txt`** — Plain text transcript, one line per spoken segment
> - **`.srt`** — Subtitles with timestamps (for video players, Premiere, DaVinci Resolve, etc.)
> - **`.vtt`** — Web subtitles format (for YouTube, HTML5 video, etc.)
>
> The output filename always matches your source file:
> `Interview.mp3` → `Interview.txt`, `Interview.srt`, `Interview.vtt`

**⑥ Everything zipped**
> Once all files are transcribed, a zip file containing every `.txt`, `.srt`, and `.vtt` is saved to:
> ```
> /content/all_transcribed_files.zip
> ```
> **Download this zip immediately** — all files in `/content/` are erased when the Colab session ends.

**⑦ Final summary**
> A report prints at the end:
> ```
> ====================================================
> TRANSCRIPTION SUMMARY
>   Total files :  5
>   Succeeded   :  5
>   Failed      :  0
> ====================================================
> ```
> If any file failed, the reason is listed here.

---

## 📥 Downloading Your Results

1. Find `all_transcribed_files.zip` in the Files panel under `/content/`
2. Right-click it → **Download**

> ⚠️ **Important:** Colab sessions expire after a period of inactivity. Download your zip as soon as transcription finishes — files cannot be recovered after the session ends.

---

## ❓ Troubleshooting

| Issue | Fix |
|-------|-----|
| *"No supported audio/video files found"* | Make sure files are inside `/content/bulk_process_audios_here`, not in a subfolder |
| *"ffmpeg binary not found"* | Add a new code cell above with `!apt install -y ffmpeg` and run it first |
| *Transcription is extremely slow* | Switch to a GPU runtime: **Runtime → Change runtime type → T4 GPU** |
| *GPU out of memory* | Restart the runtime (**Runtime → Restart session**) and re-run the cell |
| *File skipped with [SKIP]* | The file is either empty (0 bytes) or unreadable — re-upload it |

In [ ]:
# @title ## Whisper Bulk Transcriber
# @markdown ### Run this cell to install dependencies, detect audio files, and transcribe everything in `/content/bulk_process_audios_here`.
# @markdown Outputs `.txt`, `.srt`, and `.vtt` per file. Final zip saved to `/content/all_transcribed_files.zip`.

# ── Phase 0: Constants & Minimal Imports ─────────────────────────────────────
# Only stdlib modules needed here — no pip installs required for directory check.

import os
import sys

BULK_DIR   = "/content/bulk_process_audios_here"
OUTPUT_DIR = "/content/audio_transcription"
ZIP_PATH   = "/content/all_transcribed_files.zip"
USE_MODEL  = "large-v2"

VIDEO_EXTENSIONS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

SUPPORTED_EXTENSIONS = {
    '.mp3', '.wav', '.m4a', '.flac', '.aac', '.ogg', '.wma',
    '.aiff', '.opus', '.amr', '.au',
} | VIDEO_EXTENSIONS

TRANSCRIBED_EXTS = {'.txt', '.srt', '.vtt'}

# ── Phase 1: Directory Check (runs FIRST — before any installs) ───────────────
# Creating the upload directory and verifying files are present is instant and
# requires no external dependencies.  Failing fast here saves the user from
# waiting through pip installs and model loading when there is nothing to do.

os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(BULK_DIR):
    os.makedirs(BULK_DIR)
    sys.exit(
        f"\nDirectory created: '{BULK_DIR}'\n"
        f"Upload your audio/video files there, then run this cell again.\n"
    )

_all_in_dir = [
    os.path.join(BULK_DIR, f) for f in os.listdir(BULK_DIR)
    if os.path.isfile(os.path.join(BULK_DIR, f))
]
audio_files = sorted([
    f for f in _all_in_dir
    if os.path.splitext(f)[1].lower() in SUPPORTED_EXTENSIONS
])

if not audio_files:
    sys.exit(
        f"\nNo supported audio/video files found in '{BULK_DIR}'.\n"
        f"Supported formats: {', '.join(sorted(SUPPORTED_EXTENSIONS))}\n"
        f"Upload files there and run this cell again.\n"
    )

print(f"Found {len(audio_files)} file(s) to transcribe. Loading dependencies...")

# ── Phase 2: Full Imports ─────────────────────────────────────────────────────
# Only reached when files are confirmed present.

import re
import subprocess
import tempfile
import zipfile
import unicodedata

# ── Phase 3: Library Check / Install ─────────────────────────────────────────

def _install_libraries():
    cmd = [
        sys.executable, "-m", "pip", "install", "-q",
        "--root-user-action=ignore",
        "openai-whisper", "openai", "pydub", "ffmpeg-python"
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("[ERROR] pip install failed:")
        print(result.stderr)
        sys.exit(1)

try:
    import whisper
    import torch
    import pydub   # noqa: F401
    import ffmpeg  # noqa: F401  (package: ffmpeg-python, imports as ffmpeg)
    print("Libraries already installed. Skipping installation.")
except ImportError:
    print("Installing required libraries...")
    _install_libraries()
    try:
        import whisper
        import torch
        import pydub   # noqa: F401
        import ffmpeg  # noqa: F401
        print("Libraries installed and imported successfully.")
    except ImportError as e:
        print(f"[ERROR] Import failed after installation: {e}")
        sys.exit(1)

# ── Phase 4: ffmpeg Binary Pre-flight Check ───────────────────────────────────

_ffmpeg_check = subprocess.run(
    ["ffmpeg", "-version"], capture_output=True
)
if _ffmpeg_check.returncode != 0:
    print("[ERROR] ffmpeg binary not found on PATH.")
    print("Fix: add a cell above with   !apt install -y ffmpeg   and run it first.")
    sys.exit(1)
print("ffmpeg binary confirmed.")

# ── Phase 5: Per-file Pre-validation ─────────────────────────────────────────

valid_audio_files = []
for _f in audio_files:
    _name = os.path.basename(_f)
    if not os.access(_f, os.R_OK):
        print(f"[SKIP] Not readable (check permissions): {_name}")
        continue
    if os.path.getsize(_f) == 0:
        print(f"[SKIP] Empty file (0 bytes): {_name}")
        continue
    valid_audio_files.append(_f)

if not valid_audio_files:
    sys.exit("[ERROR] No valid audio files remain after validation. Aborting.")

audio_files = valid_audio_files
print(f"{len(audio_files)} file(s) passed validation.")

# ── Phase 6: Model Load ───────────────────────────────────────────────────────

from whisper.utils import get_writer  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cpu":
    # Probe total audio duration so the warning can show a time estimate.
    _total_secs = 0.0
    _duration_known = True
    for _af in audio_files:
        _probe = subprocess.run(
            [
                "ffprobe", "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                _af,
            ],
            capture_output=True, text=True,
        )
        try:
            _total_secs += float(_probe.stdout.strip())
        except ValueError:
            _duration_known = False
            break

    # large-v2 on a typical Colab CPU runs ~20x slower than real-time.
    _EST_SLOWDOWN = 20
    _border = "!" * 64
    print(f"\n{_border}")
    print("  WARNING: NO GPU DETECTED — TRANSCRIPTION WILL BE")
    print("  OVERWHELMINGLY SLOW (roughly 1 transcribed line per minute).")
    print()
    if _duration_known and _total_secs > 0:
        _audio_mins = _total_secs / 60
        _est_mins   = (_total_secs * _EST_SLOWDOWN) / 60
        _est_hrs    = _est_mins / 60
        print(f"  Total audio duration : ~{_audio_mins:.0f} min")
        if _est_hrs >= 1:
            print(f"  Estimated CPU time   : ~{_est_hrs:.1f} hours  (could be longer)")
        else:
            print(f"  Estimated CPU time   : ~{_est_mins:.0f} minutes  (could be longer)")
        print()
    print("  STRONGLY RECOMMENDED: switch to a GPU runtime first.")
    print("  Runtime > Change runtime type > Hardware accelerator > T4 GPU")
    print(f"{_border}\n")

    _answer = input(
        "Continue on CPU anyway? This will take an extremely long time. [y/N]: "
    ).strip().lower()

    if _answer != "y":
        sys.exit(
            "\nAborted. To enable GPU:\n"
            "  Runtime > Change runtime type > Hardware accelerator > T4 GPU\n"
            "Then run this cell again.\n"
        )
    print("Acknowledged. Continuing on CPU — this will take a very long time.\n")

print(f"Loading '{USE_MODEL}' model on {DEVICE}...")
try:
    model = whisper.load_model(USE_MODEL, device=DEVICE)
    print(f"Model '{USE_MODEL}' loaded on {DEVICE}.")
except Exception as e:
    print(f"[ERROR] Failed to load Whisper model: {e}")
    sys.exit(1)

# ── Phase 7: LiveFileWriter ───────────────────────────────────────────────────
#
# Streams Whisper's verbose console output in real-time AND writes transcript
# lines to a .txt file, keeping a single file handle open for the full
# duration of each transcription (one open/close per file, not per segment).
#
# Segment detection uses a regex anchored to Whisper's exact verbose format:
#   [MM:SS.mmm --> MM:SS.mmm]  transcribed text here
# This prevents false positives when the spoken content contains " --> ".

_WHISPER_SEGMENT_RE = re.compile(
    r'^\[\d{2}:\d{2}\.\d{3} --> \d{2}:\d{2}\.\d{3}\]'
)


class LiveFileWriter:
    """Tees sys.stdout to the console and to a UTF-8 .txt file simultaneously."""

    def __init__(self, stream, file_path: str):
        self.stream = stream
        self._file = open(file_path, "a", encoding="utf-8")

    def write(self, data):
        if data is None:
            return
        self.stream.write(data)
        self.stream.flush()
        # Only write lines that match a Whisper transcript segment header.
        if _WHISPER_SEGMENT_RE.match(data.strip()):
            # Extract text after the closing ']' of the timestamp pair.
            text_only = data.split("] ", 1)[-1].strip()
            if text_only:
                self._file.write(text_only + "\n")
                self._file.flush()

    def writelines(self, datas):
        for d in datas:
            self.write(d)

    def close(self):
        if self._file and not self._file.closed:
            self._file.close()

    def __getattr__(self, name):
        return getattr(self.stream, name)


# ── Phase 7b: Video Audio Extraction Helper ───────────────────────────────────
#
# For video files, ffmpeg strips the video stream and writes a 16 kHz mono
# PCM WAV to a temporary file — the exact format Whisper loads natively,
# avoiding any re-decode overhead inside whisper.load_audio().
#
# The temp file is always deleted in the transcription loop's finally block,
# even if transcription fails or raises an exception.

def _extract_audio_from_video(video_path: str) -> str:
    """Extract audio track from a video file to a temp WAV; return its path."""
    tmp = tempfile.NamedTemporaryFile(
        suffix=".wav",
        prefix="_whisper_extract_",
        delete=False,
    )
    tmp_path = tmp.name
    tmp.close()  # close so ffmpeg can write to it on all platforms

    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vn",               # discard video stream
        "-acodec", "pcm_s16le",  # 16-bit signed PCM — Whisper's native format
        "-ar", "16000",      # 16 kHz sample rate
        "-ac", "1",          # mono
        tmp_path,
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass
        # Surface the last line of stderr as the error message.
        err_detail = proc.stderr.strip().splitlines()[-1] if proc.stderr.strip() else "unknown ffmpeg error"
        raise RuntimeError(f"Audio extraction failed: {err_detail}")

    return tmp_path


# ── Phase 8: Transcription Options ───────────────────────────────────────────

_options = {
    "task": "transcribe",
    "fp16": DEVICE == "cuda",
    "best_of": 5,
    "beam_size": 5,
    "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
    "condition_on_previous_text": True,
    "initial_prompt": None,
    "language": None,               # None = auto-detect
    # Sensitivity tuning (quieter speech detection)
    "no_speech_threshold": 0.4,     # default 0.6  — lower = less skipping
    "logprob_threshold": -1.5,      # default -1.0 — lower = keep low-confidence
    "compression_ratio_threshold": 2.2,  # default 2.4 — lower = avoid repetitions
}

# ── Phase 9: Transcription Loop ──────────────────────────────────────────────

results      = {}   # audio_path -> whisper result dict (successes)
failed_files = {}   # audio_path -> error message (failures)
original_stdout = sys.stdout
n = len(audio_files)

for i, audio_path in enumerate(audio_files, 1):
    # Derive output stem; NFC-normalise for cross-platform zip safety.
    # macOS uploads use NFD decomposition; Linux/Windows zip extraction needs NFC.
    raw_stem    = os.path.splitext(os.path.basename(audio_path))[0]
    output_stem = unicodedata.normalize("NFC", raw_stem)
    txt_path    = os.path.join(OUTPUT_DIR, output_stem + ".txt")

    print(f"\n[{i}/{n}] Processing: {os.path.basename(audio_path)}")

    # --- Video → audio extraction -------------------------------------------
    # If the source file is a video, extract its audio track to a temp WAV
    # before passing to Whisper.  Output filenames are always derived from the
    # original source filename (audio_path), never from the temp file.
    _temp_audio = None
    transcribe_path = audio_path

    if os.path.splitext(audio_path)[1].lower() in VIDEO_EXTENSIONS:
        print(f"  Video file detected — extracting audio track via ffmpeg...")
        try:
            _temp_audio    = _extract_audio_from_video(audio_path)
            transcribe_path = _temp_audio
            print(f"  Audio extracted successfully.")
        except RuntimeError as e:
            msg = str(e)
            print(f"[FAIL] {os.path.basename(audio_path)}: {msg}")
            failed_files[audio_path] = msg
            continue  # skip to next file; no temp file was created on failure

    # Clear any previous .txt for this file before starting.
    with open(txt_path, "w", encoding="utf-8") as _f:
        _f.write("")

    live_writer = LiveFileWriter(original_stdout, txt_path)
    result = None

    try:
        sys.stdout = live_writer
        result = whisper.transcribe(model, transcribe_path, verbose=True, **_options)

    except RuntimeError as e:
        msg = str(e)
        if "out of memory" in msg.lower():
            torch.cuda.empty_cache()
            msg = (
                "GPU out of memory. "
                "Try a shorter audio file, or restart the runtime to free VRAM: "
                "Runtime > Restart session."
            )
        else:
            msg = f"RuntimeError: {e}"
        failed_files[audio_path] = msg
        print(f"\n[FAIL] {os.path.basename(audio_path)}: {msg}")

    except Exception as e:
        msg = str(e)
        failed_files[audio_path] = msg
        print(f"\n[FAIL] {os.path.basename(audio_path)}: {msg}")

    finally:
        # Always restore stdout, close the file handle, and delete any temp
        # audio file — regardless of whether transcription succeeded or failed.
        sys.stdout = original_stdout
        live_writer.close()
        if _temp_audio and os.path.exists(_temp_audio):
            os.unlink(_temp_audio)

    if result is None:
        continue  # transcription failed; move to next file

    # Free GPU memory between files to reduce OOM risk on large batches.
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    if not result.get("segments"):
        print(f"  No speech detected in: {os.path.basename(audio_path)}")

    results[audio_path] = result
    print(f"  Saved: {output_stem}.txt")

    # Write .srt and .vtt using Whisper's built-in writers.
    # Pass a NFC-normalised path so the writer derives a clean output filename.
    norm_audio_path = os.path.join(
        os.path.dirname(audio_path),
        output_stem + os.path.splitext(audio_path)[1]
    )

    for fmt in ("srt", "vtt"):
        try:
            # VTT spec rejects a first cue that starts at exactly 00:00.000.
            # Offset by 1 ms, write, then restore the original value.
            fix_vtt = (
                fmt == "vtt"
                and bool(result.get("segments"))
                and result["segments"][0].get("start") == 0
            )
            if fix_vtt:
                result["segments"][0]["start"] += 1 / 1000

            writer = get_writer(fmt, OUTPUT_DIR)
            writer(result, norm_audio_path)

            if fix_vtt:
                result["segments"][0]["start"] = 0

            print(f"  Saved: {output_stem}.{fmt}")

        except Exception as e:
            print(f"  [WARN] Could not write .{fmt} for '{output_stem}': {e}")

# ── Phase 10: Zip All Transcribed Files ──────────────────────────────────────

if not results:
    print("\n[ERROR] No files transcribed successfully. Nothing to zip.")
else:
    _files_to_zip = sorted([
        os.path.join(OUTPUT_DIR, fname)
        for fname in os.listdir(OUTPUT_DIR)
        if os.path.splitext(fname)[1].lower() in TRANSCRIBED_EXTS
    ])

    if not _files_to_zip:
        print("\n[WARN] Output directory has no .txt/.srt/.vtt files to zip.")
    else:
        with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
            for fpath in _files_to_zip:
                # arcname stores only the filename (no subdirectory prefix) in the zip.
                zf.write(fpath, arcname=os.path.basename(fpath))

        _zip_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
        print(
            f"\nZipped {len(_files_to_zip)} file(s) -> {ZIP_PATH} "
            f"({_zip_mb:.2f} MB)"
        )
        print(
            "REMINDER: Download your zip before the Colab session ends. "
            "/content/ is erased when the runtime disconnects."
        )

# ── Phase 11: Final Summary ───────────────────────────────────────────────────

print("\n" + "=" * 52)
print("TRANSCRIPTION SUMMARY")
print(f"  Total files : {n}")
print(f"  Succeeded   : {len(results)}")
print(f"  Failed      : {len(failed_files)}")
if failed_files:
    print("\nFailed files:")
    for _path, _reason in failed_files.items():
        print(f"  x {os.path.basename(_path)}")
        print(f"    Reason: {_reason}")
print("=" * 52)
